## 📑 Core Extrajudicial Letter
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/CID5Gg)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
# Databricks notebook source
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
%python
table_name = 'Core_ExrtajudicialLetter'

schema = (
"BusinessDate date,"
"PortfolioName string,"
"AgreementKey string,"
"AgreementKeyClean string,"
"ExposureKey string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"PartyRelationType string,"
"ActivityTypeKey string,"
"ActivityName string,"
"ActivityDate date,"
"LetterActivityDate date,"
"Source string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Optimizer Function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    


##### ExtrajudicialLetterHistoricalCore

In [0]:
%python
core_extrajudicialletterhistorical = spark.sql(
f'''
select * from  `{env_prefix}-reportinputfiles`.repoint.core_extrajudicialletterhistorical 
'''
)
core_extrajudicialletterhistorical.createOrReplaceTempView("core_extrajudicialletterhistorical")

##### dexposure 

In [0]:
%python
dexposure_active = spark.sql(
f'''
SELECT 
    A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    ,A.AgreementKey
    ,A.IsActiveFlag
    ,A.ProductKey
FROM  `{env_prefix}-cdh`.derivative.dexposure as A  
where A.BusinessDate = {run_date!r}
and SourceSystem ='qca'
and PortfolioName = 'Core Retail'
and IsActiveFlag = 'true'
'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")



dexposure_inactive = spark.sql(
f'''
SELECT
    {run_date!r} as BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    ,coalesce(A.AgreementKey, B.AgreementKey) as AgreementKey
    ,false as IsActiveFlag
    ,A.ProductKey
from (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      FROM  `{env_prefix}-cdh`.derivative.dexposure as e
      where 1=1
        and  e.SourceSystem = 'qca'
        and e.BusinessDate <= {run_date!r}
        and  e.PortfolioName = 'Core Retail'
        ) AS A
left join (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      FROM  `{env_prefix}-cdh`.derivative.dexposure as e -- `{env_prefix}-cdh`.primary.exposure as e 
      where 1=1
        and  e.SourceSystem = 'qca'
        and e.BusinessDate <= {run_date!r}
        and  e.PortfolioName = 'Core Retail'
        and e.AgreementKey is not null
        ) AS B
on a.exposurekey = b.exposurekey and b.rank = 1
left anti join dexposure_active e_pf_act on e_pf_act.ExposureKey = A.ExposureKey and e_pf_act.BusinessDate = {run_date!r}
where A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")



Dexposure = spark.sql(
f'''
select *
from dexposure_active
 
union all
 
select *
from dexposure_inactive
 
'''
)
Dexposure.createOrReplaceTempView("Dexposure")

##### exposure

In [0]:
%python
exposure = spark.sql(
f'''
select distinct e.CDI, e.PartyTaxRegistrationNumber as PartyTaxRegistrationNumber  
from  `{env_prefix}-cdh`.primary.exposure e
where portfolioName = 'Core Retail' and e.PartyTaxRegistrationNumber is not null
UNION 
select distinct  CDI, PartyTaxRegistrationNumber
from `{env_prefix}-cdh`.derivative.dexposure  d
where portfolioName = 'Core Retail' and PartyTaxRegistrationNumber is not null

'''
)
exposure.createOrReplaceTempView("exposure")

##### Product

In [0]:
%python
Product = spark.sql(
f'''
select distinct
    ProductKey
    ,BankingUnitCode as SubProduct
from `{env_prefix}-cdh`.primary.product
WHERE SourceSystem ='qca' and BusinessDate = {run_date!r}
'''
)
Product.createOrReplaceTempView("Product")

## NonCorePerimeter

In [0]:
%python
NonCorePerimeter = spark.sql(
f'''
    select distinct CASE
        WHEN LEFT(Exposurekey, 1) BETWEEN 'A' AND 'Z' THEN  
            SUBSTRING(Exposurekey, 4, LEN(Exposurekey) - 3)
        ELSE
            ExposureKey
    END AS Exposurekey
    , portfolioname
    from `{env_prefix}-cdh`.primary.exposure
    where PortfolioName <> 'Core Retail'
    and BusinessDate <= {run_date!r}
    and portfolioname in (select distinct portfolioname
    from `{env_prefix}-cdh`.primary.exposure
    where PortfolioName <> 'Core Retail'
    and BusinessDate <= {run_date!r}
    and sourcesystem = 'qca'
    and portfolioname is not null)
'''
)
NonCorePerimeter.createOrReplaceTempView("NonCorePerimeter")

# create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_NonCorePerimeter", NonCorePerimeter)
# repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_NonCorePerimeter")

##### MaxBusinessDates

In [0]:
%python
MaxBusinessDates = spark.sql(
f'''
select mbb.ExposureKey, mbb.MaxBusinessDate, c.NextBusinessDay NextBusinessDayOfMaxBusinessDate 
from (
select ExposureKey, max(BusinessDate) maxBusinessDate
from `{env_prefix}-cdh`.primary.exposure 
where PortfolioName = 'Core Retail' and IsActiveFlag = 'true'
group by ExposureKey
) mbb
inner join `{env_prefix}-cdh`.reference.ref_calendar c
on mbb.maxBusinessDate = c.CalendarDate 
'''
)
MaxBusinessDates.createOrReplaceTempView("MaxBusinessDates")

##### core_extrajudicialinteraction

In [0]:
%python
core_extrajudicialinteraction = spark.sql(
f'''
select distinct 
 l.ExposureKey 
,l.InteractionKey
,l.CDI 
,l.PortfolioName 
,l.InteractionDate 
,l.InteractionTypeKey 
,l.InteractionType
FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
where 1=1
and InteractionDate <= {run_date!r}
and l.PortfolioName = 'Core Retail'
and l.SourceSystem = 'qca'
and InteractionTypeKey in ('2403', '2404', '2405', '1164', '2407', '2408', '2410')

UNION 

select distinct 
 c.ExposureKey 
,l.InteractionKey
,coalesce(c.CDI, l.CDI) as CDI
-- , l.CDI
,l.PortfolioName 
,l.InteractionDate 
,l.InteractionTypeKey 
,l.InteractionType 
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
inner join (
    select distinct c.casecodekey, e.Exposurekey, e.CDI
    from `{env_prefix}-cdh`.primary.casecode c
    inner join (
    select distinct caseCode, Exposurekey, CDI
    from `{env_prefix}-cdh`.primary.exposure 
    where PortfolioName = 'Core Retail'
    ) e
    on c.casecodekey = e.casecode
    where 1=1 
    and c.PortfolioName = 'Core Retail'
) c
on l.casecode = c.casecodekey
left join MaxBusinessDates as mbb 
on mbb.ExposureKey = c.ExposureKey
where 1=1
and InteractionTypeKey in ('2403', '2404', '2405', '1164', '2407', '2408', '2410')
and InteractionDate <= {run_date!r}
and l.SourceSystem = 'qca'
and l.InteractionDate <= ifnull(mbb.MaxBusinessDate, '9999-12-30')
'''
)
core_extrajudicialinteraction.createOrReplaceTempView("core_extrajudicialinteraction")

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_core_extrajudicialinteraction", core_extrajudicialinteraction)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_core_extrajudicialinteraction")

##### QualcoPerimeterCalculated

In [0]:
%python
step1 = spark.sql(
f'''
SELECT distinct 
        A.ExposureKey,
       A.AgreementKey,
        A.PortfolioName,
       CASE 
           WHEN D.SubProduct LIKE '25%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '01%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '23%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '02%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '36%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '37%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '18%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '03%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '%ΣΣΕ84%' THEN cast(A.AgreementKey as string)
           WHEN LEN(A.AgreementKey)=10 THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           ELSE cast(A.AgreementKey as string)
       END AS AgreementKeyClean,
       A.IsActiveFlag AS IsActiveFlag
FROM Dexposure A
left join Product D on A.ProductKey = D.ProductKey
'''
)
step1.createOrReplaceTempView("step1")

In [0]:
%python
QualcoPerimeterCalculated = spark.sql(
f'''
SELECT
ExposureKey
,AgreementKey
,AgreementKeyClean
,PortfolioName
,IsActiveFlag
FROM step1
'''
)
QualcoPerimeterCalculated.createOrReplaceTempView("QualcoPerimeterCalculated")

# create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_QualcoPerimeterCalculated", QualcoPerimeterCalculated)
# repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_QualcoPerimeterCalculated")

##### AllActivities

In [0]:
%python
AllActivities = spark.sql(
f'''
select distinct 
 l.ExposureKey
,l.InteractionKey
,l.CDI
,l.PortfolioName
,l.InteractionDate as ActivityDate
,l.InteractionTypeKey as ActivityTypeKey
,l.InteractionType as ActivityName
,d.PartyTaxRegistrationNumber
FROM `{env_prefix}-cdh-delivery`.repoint.int_core_extrajudicialinteraction l
left join exposure d
on d.cdi = l.cdi
'''
)
AllActivities.createOrReplaceTempView("AllActivities")


##### Cancellations

In [0]:
%python
Cancellations = spark.sql(
f'''
SELECT 
					A.*
		FROM AllActivities a
		WHERE A.ActivityTypeKey IN ('2407', '2408', '2410')
'''
)
Cancellations.createOrReplaceTempView("Cancellations")

##### Eksodika

In [0]:
%python
Eksodika = spark.sql(
f'''
SELECT 
					A.*
		FROM AllActivities a
		WHERE A.ActivityTypeKey IN ('2403', '2404', '2405')
'''
)
Eksodika.createOrReplaceTempView("Eksodika")

##### 1164

In [0]:
%python
table_1164 = spark.sql(
f'''
SELECT 
					A.*
		FROM AllActivities a
		WHERE A.ActivityTypeKey = '1164'
'''
)
table_1164.createOrReplaceTempView("table_1164")

##### QualcoCancellations

In [0]:
%python
QualcoCancellations = spark.sql(
f'''
SELECT DISTINCT
					E.*
		FROM Eksodika AS E
		LEFT JOIN Cancellations AS C ON  E.ExposureKey = C.ExposureKey
										  AND E.CDI = C.CDI
		WHERE 1=1
			  AND DATEDIFF(day, C.ActivityDate, E.ActivityDate) < 0
			  AND DATEDIFF(day, C.ActivityDate, E.ActivityDate) > -60
'''
)
QualcoCancellations.createOrReplaceTempView("QualcoCancellations")

##### HistoricalCancellations

In [0]:
%python
HistoricalCancellations = spark.sql(
f'''
SELECT DISTINCT
					H.*
		FROM Core_ExtrajudicialLetterHistorical	AS H
		LEFT JOIN Cancellations AS C ON  H.Account_Number = C.ExposureKey 
										  AND ( H.CIF = C.CDI OR
												H.AFM = C.PartyTaxRegistrationNumber)
		WHERE 1=1
			  AND DATEDIFF(day, C.ActivityDate, H.ACTIVITY_DATE) < 0
			  AND DATEDIFF(day, C.ActivityDate, H.ACTIVITY_DATE) > -60
'''
)
HistoricalCancellations.createOrReplaceTempView("HistoricalCancellations")

##### CustomerInfo

In [0]:
%python
CustomerInfo = spark.sql(
f'''
select 
DISTINCT 
      ExposureKey
      ,CDI
      ,PartyRelationshipType
      --,CASE WHEN PartyRelationshipType = 'Primary Owner' THEN 'loan-Main Beneficiary Rltp'
      --      WHEN PartyRelationshipType = 'Co-Owner' THEN 'Loan-Other Beneficiaries Rltp'
      --      WHEN PartyRelationshipType = 'Guarantor' THEN 'Loan-Guarantor Rltp'
      --      ELSE PartyRelationshipType 
      --END AS RelationType  /*** No need for case when during deployment, just PartyRelationshipType ***/
from  (
      select distinct ExposureKey, CDI, PartyRelationshipType, rank() over (partition by ExposureKey order by BusinessDate desc) as rank
      from `{env_prefix}-cdh`.primary.exposure
      where portfolioName = 'Core Retail' and PartyRelationshipType is not null
)
where rank = 1
'''
)
CustomerInfo.createOrReplaceTempView("CustomerInfo")
# create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_CustomerInfo", CustomerInfo)
# repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_CustomerInfo")

##### ExtraJudicialLetter

In [0]:
%python
ExtraJudicialLetter = spark.sql(
f'''
SELECT	DISTINCT	t.BusinessDate
					,	t.PortfolioName
					,	t.AgreementKey
					,	t.AgreementKeyClean
					,	t.ExposureKey
					,	t.CDI
					,	CASE WHEN t.PartyTaxRegistrationNumber IN ('0000','000000000', '999999999', '') THEN NULL ELSE t.PartyTaxRegistrationNumber END		AS PartyTaxRegistrationNumber
					,	t.PartyRelationType
					,	t.ActivityTypeKey
					,	t.ActivityName
					,	t.ActivityDate
					,	t.LetterActivityDate
					,	t.Source	
		FROM (
			
    	SELECT DISTINCT
						{run_date!r}			AS BusinessDate
					,	E.ExposureKey
					,	A.PortfolioName
					,	A.AgreementKey
					,	A.AgreementKeyClean
					,	E.CDI
					,	coalesce(E.PartyTaxRegistrationNumber, H.AFM) as PartyTaxRegistrationNumber
					,	coalesce(CI.PartyRelationshipType, H2.Relationship_Type) as PartyRelationType
					,	E.ActivityTypeKey
					,	E.ActivityName
					,	E.ActivityDate
					,	NULL					AS LetterActivityDate
					,	'Qualco'				AS Source
			FROM Eksodika AS E
			INNER JOIN QualcoPerimeterCalculated AS A ON A.ExposureKey = E.ExposureKey 
			LEFT JOIN CustomerInfo AS CI ON CI.ExposureKey = E.ExposureKey AND CI.CDI = E.CDI
			--INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_QualcoPerimeterCalculated AS A ON A.AccountNumber = E.AccountNumber 
			--LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_CustomerInfo AS CI ON CI.AccountNumber = E.AccountNumber AND CI.DebtorCDI = E.DebtorCDI
			LEFT JOIN QualcoCancellations AS CC ON E.ExposureKey = CC.ExposureKey AND
														E.CDI = CC.CDI AND
														E.ActivityTypeKey = CC.ActivityTypeKey AND
														E.ActivityDate = CC.ActivityDate 
              
			LEFT JOIN (
			SELECT DISTINCT e1.Account_Number, e1.CIF, CASE WHEN e1.AFM IN ('0000','000000000', '999999999', '') THEN NULL ELSE e1.AFM END as AFM, e1.Activity_Date, e1.Activity_Code 
			from core_extrajudicialletterhistorical e1
			inner join  
			(
				select distinct Account_Number, CIF, Activity_Date, activity_Code
				from core_extrajudicialletterhistorical
				where afm is not null
				group by Account_Number, CIF, Activity_Date, Activity_Code
				having count(distinct AFM) = 1
			) e2	
			on e1.Account_Number = e2.Account_Number
			and ifnull(e1.CIF, 'NULL') = ifnull(e2.CIF, 'NULL')
			and e1.Activity_Date = e2.Activity_Date
			and e1.Activity_Code = e2.Activity_Code
			where e1.afm is not null
		) H
		ON  E.ExposureKey = H.ACCOUNT_NUMBER	
		AND H.CIF = E.CDI 
		AND E.ActivityDate = H.ACTIVITY_DATE
		AND E.ActivityTypeKey = H.ACTIVITY_CODE  
	 LEFT JOIN (
					SELECT DISTINCT e1.Account_Number, e1.CIF, e1.Relationship_Type, e1.Activity_Date, e1.Activity_Code 
			from core_extrajudicialletterhistorical e1
			inner join  
			(
				select distinct Account_Number, CIF, Activity_Date, activity_Code
				from core_extrajudicialletterhistorical
				where Relationship_Type is not null
				group by Account_Number, CIF, Activity_Date, Activity_Code
				having count(distinct Relationship_Type) = 1
			) e2	
			on e1.Account_Number = e2.Account_Number
			and ifnull(e1.CIF, 'NULL') = ifnull(e2.CIF, 'NULL')
			and e1.Activity_Date = e2.Activity_Date
			and e1.Activity_Code = e2.Activity_Code
			where e1.Relationship_Type is not null
			) H2
		ON  E.ExposureKey = H2.ACCOUNT_NUMBER	
	AND H2.CIF = E.CDI 
	AND E.ActivityDate = H2.ACTIVITY_DATE
	AND E.ActivityTypeKey = H2.ACTIVITY_CODE  
	WHERE 1=1
			AND CC.ExposureKey IS NULL
			
			UNION 

			SELECT DISTINCT 
						{run_date!r}			AS BusinessDate
					,	h.ACCOUNT_NUMBER
					,	'Core Retail' AS PortfolioName
					,	h.CASE_CONTR_NUM
					,	h.CASE_CONTR_NUM
					,	H.CIF
					,	H.AFM
					,	H.RELATIONSHIP_TYPE
					,	H.ACTIVITY_CODE
					,	H.ACTIVITY_NAME
					,	H.ACTIVITY_DATE
					,	NULL					AS LetterActivityDate
					,	H.Source
			FROM Core_ExtrajudicialLetterHistorical		AS H 
			LEFT JOIN 
   			(SELECT E.ExposureKey, E.CDI, E.PartyTaxRegistrationNumber, E.ActivityDate, E.ActivityTypeKey from 
      		eksodika E 
			inner join QualcoPerimeterCalculated cp 
			on e.ExposureKey = cp.ExposureKey

			--inner join `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_QualcoPerimeterCalculated cp 
			--on e.AccountNumber = cp.AccountNumber
			) AS E 
   			ON  E.ExposureKey = H.ACCOUNT_NUMBER	
								AND H.CIF = E.CDI 
								AND E.ActivityDate = H.ACTIVITY_DATE
								AND E.ActivityTypeKey = H.ACTIVITY_CODE
			LEFT JOIN HistoricalCancellations AS CC ON H.ACCOUNT_NUMBER = CC.ACCOUNT_NUMBER AND
															ifnull(H.CIF,'') = ifnull(CC.CIF,'') AND 
															ifnull(H.AFM,'') = ifnull(CC.AFM,'') AND 
															H.ACTIVITY_CODE = CC.ACTIVITY_CODE AND
															H.ACTIVITY_DATE = CC.ACTIVITY_DATE
			WHERE 1=1 
				  AND E.ExposureKey	IS NULL 
				  AND CC.ACCOUNT_NUMBER IS NULL
		
		UNION

			SELECT DISTINCT
						{run_date!r}			AS BusinessDate
					,	E.ExposureKey
					, A.PortfolioName
					,	A.AgreementKey
					,	A.AgreementKeyClean
					,	E.CDI
					,	E.PartyTaxRegistrationNumber
					,	CI.PartyRelationshipType
					,	E.ActivityTypeKey
					,	E.ActivityName
					,	NULL					AS ActivityDate
					,	E.ActivityDate		AS LetterActivityDate
					,	'Qualco'				AS Source
			FROM table_1164 AS E
			INNER JOIN QualcoPerimeterCalculated AS A ON A.ExposureKey = E.ExposureKey 
			LEFT JOIN CustomerInfo AS CI ON CI.ExposureKey = E.ExposureKey AND CI.CDI = E.CDI

			--INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_QualcoPerimeterCalculated AS A ON A.AccountNumber = E.AccountNumber 
			--LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_CustomerInfo AS CI ON CI.AccountNumber = E.AccountNumber AND CI.DebtorCDI = E.DebtorCDI
			
		) AS t
left join NonCorePerimeter ncp
--left join `{env_prefix}-cdh-delivery`.repoint.int_ra_extrajudicial_NonCorePerimeter ncp
on t.ExposureKey = ncp.ExposureKey 
WHERE NCP.ExposureKey IS NULL
'''
)
ExtraJudicialLetter.createOrReplaceTempView("ExtraJudicialLetter")

In [0]:
%python
final_df = spark.sql(
f'''
SELECT	cast(BusinessDate as date) AS BusinessDate
	,	cast(PortfolioName as string) AS PortfolioName
	,	cast(AgreementKey as string) AS AgreementKey
	,	cast(AgreementKeyClean as string) AS AgreementKeyClean
	,	cast(ExposureKey as string) AS ExposureKey
	,	cast(CDI as string) AS CDI
	,	cast(PartyTaxRegistrationNumber as string) AS PartyTaxRegistrationNumber 
  ,	cast(PartyRelationType as string) AS PartyRelationType
	,	cast(ActivityTypeKey as string) AS ActivityTypeKey
	,	cast(ActivityName as string) AS ActivityName
	,	cast(ActivityDate as date) AS ActivityDate
	,	cast(LetterActivityDate as date) AS LetterActivityDate
	,	cast(Source as string) AS Source
  , cast(getdate() as timestamp) as InsertTimeStamp 
FROM ExtraJudicialLetter
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
%python
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_ExtrajudicialLetter: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
%python
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
%python
extrajudicial_letter = spark.sql(
f''' 
SELECT * 
FROM {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
extrajudicial_letter.createOrReplaceTempView("extrajudicial_letter")

In [0]:
%python
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/Extrajudicial_Letter/"
file_name = "Extrajudicial_Letter_" +run_date+ ".csv"
extrajudicial_letter.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)